## Version 1

## INIT

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("ALS_with_meta") \
    .getOrCreate()

bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
26/03/18 07:07:54 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:07:54 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 07:07:54 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Pleas

In [2]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

date = "2026-03-17"
required_days = 90

In [3]:
daily_watch_history_path = "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new"
date_format = "%Y-%m-%d"
base_date = datetime.strptime(date, date_format)

paths = [
    f"{daily_watch_history_path}/day={(base_date - timedelta(days=i)).strftime(date_format)}"
    for i in range(required_days)
]

watch_history_df = None

for path in paths:
    try:
        temp_df = spark.read.parquet(path)

        if watch_history_df is None:
            watch_history_df = temp_df
        else:
            watch_history_df = watch_history_df.unionByName(temp_df)

    except Exception:
        print(f"Skipping missing path: {path}")

Skipping missing path: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-02-06
Skipping missing path: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-02-05


In [4]:
enriched_db_path = "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/" 

enriched_tv_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_tv.parquet')
enriched_movies_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_movie.parquet')

26/03/18 07:08:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Cleaning DB

In [5]:
filtered_enriched_tv = (
    enriched_tv_df
    # 1. Remove empty arrays and un-published content
    .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
    # 2. Explode and select in one go
    .select(
        F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
        "title", 
        "ID",
        F.col('OriginalLanguage').alias('original_language'), 
        "Genres"
    )
)
filtered_enriched_movies = (
    enriched_movies_df
    # 1. Remove empty arrays and un-published content
    .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
    # 2. Explode and select in one go
    .select(
        F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
        "title", 
        "ID", 
        F.col('OriginalLanguage').alias('original_language'),
        "Genres"
    )
)

In [6]:
# 1. Ignore files that are physically broken/incomplete
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

# 2. Disable the Vectorized Reader (Standardizes the reading process)
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# 3. Handle schema mismatches if they exist
spark.conf.set("spark.sql.parquet.mergeSchema", "true")

# 4. Perform the union with explicit casting to ensure types match
tv_final = filtered_enriched_tv.select(
    F.col("item_id").cast("string"),
    F.col("title").cast("string"),
    F.col("original_language").cast("string"),
    F.col("Genres").cast("string") # Ensure both are cast to the same type
)

movies_final = filtered_enriched_movies.select(
    F.col("item_id").cast("string"),
    F.col("title").cast("string"),
    F.col("original_language").cast("string"),
    F.col("Genres").cast("string")
)

enriched_content_df = tv_final.unionByName(movies_final).distinct()
# enriched_content_df.cache().count() # Use count() to force Spark to read and verify all files now

In [7]:
filtered_enriched_tv.show(5, truncate=False)
# filtered_enriched_movies.show(5, truncate=False)

+---------------------------------------------------------------------+--------------------------------------------+----------+-----------------+-------------------------------------------------+
|item_id                                                              |title                                       |ID        |original_language|Genres                                           |
+---------------------------------------------------------------------+--------------------------------------------+----------+-----------------+-------------------------------------------------+
|MINITV_TVSHOW_amzn1.dv.gti.510882ec-b063-435b-a448-7c2f6a998357      |JoJo no Kimyô na Bôken                      |nftv_45790|ja               |[Animation, Action & Adventure, Sci-Fi & Fantasy]|
|MINITV_TVSHOW_amzn1.dv.gti.6c59b17c-8235-4261-89dd-24ee9b70a048      |Yôkoso jitsuryoku shijô shugi no kyôshitsu e|nftv_72517|en               |[Animation, Drama, Mystery]                      |
|HOTSTAR_DTH_TVSHOW_

In [8]:
enriched_content_df = filtered_enriched_tv.unionByName(filtered_enriched_movies)
# enriched_content_df.show(10, truncate=False)

Combining playtimes

In [9]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# 1. Aggregate Playtime and Count Distinct items per user in one flow
user_stats = (
    watch_history_df  
    .groupBy("userId", "item_id")
    .agg(
        F.sum("total_play_time_sec").alias("total_playtime_combined")
    )
    # We add a column to count how many distinct items EACH user has watched
    .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId")))
)

# 2. Filter for users with at least 3 distinct items
als_input_base = user_stats.filter("distinct_content_count >= 2")

In [10]:
als_input_base.filter(F.col("userId") == "ib-xPv06AjuYOE4BI0").show(10, truncate=False)

+------------------+-----------------------------------+-----------------------+----------------------+
|userId            |item_id                            |total_playtime_combined|distinct_content_count|
+------------------+-----------------------------------+-----------------------+----------------------+
|ib-xPv06AjuYOE4BI0|LIONSGATEPLAY_MOVIE_12STRONG0Y2018M|1323.0                 |2                     |
|ib-xPv06AjuYOE4BI0|SONYLIV_VOD_MOVIE_1000131111       |1.0                    |2                     |
+------------------+-----------------------------------+-----------------------+----------------------+



## ALS

ALS data

In [11]:
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.recommendation import ALS

# 1. Clean data FIRST to avoid indexing useless rows
clean_base = als_input_base.filter(
    F.col("total_playtime_combined").isNotNull() & 
    ~F.isnan("total_playtime_combined")
)

# 2. Set up Indexers
# stringOrderType="alphabetAsc" is generally faster on massive datasets than the 
# default frequency count, which requires a heavy aggregation step.
user_indexer = StringIndexer(
    inputCol="userId", 
    outputCol="userIndex", 
    stringOrderType="alphabetAsc",
    handleInvalid="skip"
)
item_indexer = StringIndexer(
    inputCol="item_id",
    outputCol="itemIndex", 
    stringOrderType="alphabetAsc",
    handleInvalid="skip"
)

# 3. Run the Pipeline (This replaces distinct, row_number, and joins!)
pipeline = Pipeline(stages=[user_indexer, item_indexer])
indexed_data = pipeline.fit(clean_base).transform(clean_base)

item_lookup = indexed_data.select("item_id", "itemIndex").distinct()

# 4. Transform, Repartition, and Cache
als_data = (
    indexed_data
    .withColumn("playtime_logged", F.log1p("total_playtime_combined"))
    # The .cast("int") strips the heavy StringIndexer metadata out of the schema!
    .select(
        F.col("userIndex").cast("int").alias("userIndex"), 
        F.col("itemIndex").cast("int").alias("itemIndex"), 
        "playtime_logged"
    )
    .repartition(1000) 
)

als_data.cache()
als_data.show(5, truncate=False)

# 5. Train the Model
als = ALS(
    userCol="userIndex",
    itemCol="itemIndex",
    ratingCol="playtime_logged",
    implicitPrefs=True,
    rank=20,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop"
)

model = als.fit(als_data)

26/03/18 07:09:22 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:10:10 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 2 for reason Executor for container container_1764236692086_5267_01_000002 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:10:10 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 5 for reason Executor for container container_1764236692086_5267_01_000009 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:10:13 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 6 for reason Executor for container container_1764236692086_5267_01_000010 exited because of a 

+---------+---------+------------------+
|userIndex|itemIndex|playtime_logged   |
+---------+---------+------------------+
|4875979  |12912    |3.91070213346073  |
|3956731  |27589    |11.261085661561118|
|220179   |574      |8.477505137076074 |
|1736802  |33590    |7.821242083523558 |
|2567998  |528      |9.039747879981482 |
+---------+---------+------------------+
only showing top 5 rows



26/03/18 07:29:07 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 10 for reason Executor for container container_1764236692086_5267_01_000027 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:29:07 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 12 for reason Executor for container container_1764236692086_5267_01_000029 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.


Training

## Saving and loading Trained ALS model

In [ ]:

# als_model_path = "gs://wynk-ml-workspace/_temp/harshith/als/als_model"
# user_lookup_path = "gs://wynk-ml-workspace/_temp/harshith/als/user_lookup"
# item_lookup_path = "gs://wynk-ml-workspace/_temp/harshith/als/item_lookup"

# # 1. Save the ALS Model (Uses parentheses for write and overwrite)
# model.write().overwrite().save(als_model_path)

# # 2. Save the Lookup DataFrames
# # Use .mode("overwrite") instead of .overwrite
# user_lookup.write.mode("overwrite").save(user_lookup_path)
# item_lookup.write.mode("overwrite").save(item_lookup_path)

# print("✅ Models and Lookups successfully saved to GCS!")

# print("✅ Models successfully saved!")

In [13]:
# from pyspark.ml.recommendation import ALSModel

# # Load the Model
# model = ALSModel.load(als_model_path)

# # Read the Lookups
# user_lookup = spark.read.parquet(user_lookup_path)
# item_lookup = spark.read.parquet(item_lookup_path)

# print("✅ Model and Lookups loaded. Ready to recommend.")

## 

### Generating recommendations

In [14]:
user_recommendations = model.recommendForAllUsers(15)
# user_recommendations.show(10, truncate=False)

from pyspark.sql.functions import col, explode

# 1. Explode ALL recommendations for ALL users
all_recs_exploded = user_recommendations.select(
    col("userIndex"),
    explode("recommendations").alias("rec")
).select(
    "userIndex",
    col("rec.itemIndex").alias("itemIndex"),
    col("rec.rating").alias("score")
)

# 2. Perform a Global Anti-Join 
# This removes EVERY user-item pair that already exists in your training data
clean_recommendations = all_recs_exploded.join(
    als_data, # Your original training data
    on=["userIndex", "itemIndex"], 
    how="left_anti"
)

# 3. Cache this "Clean" version
# clean_recommendations.cache().show(10, truncate=False)

## Viewing Results

### Show recommendations by User

In [15]:
# from pyspark.sql.functions import col, broadcast

# # 1. Define the item_lookup correctly (Overwriting the old Window version)
# item_lookup = indexed_data.select("item_id", "itemIndex").distinct()

# # 2. Pick a random user index from the CLEANED recommendations
# random_user_idx = clean_recommendations.select("userIndex").sample(False, 0.1).limit(1).first()[0]
# print(f"--- Recommendations for User Index: {random_user_idx} ---")

# # 3. Show History (Filtering FIRST, then joining with Broadcast)
# print("\n[ History ]")
# history = (als_data
#     .filter(col("userIndex") == random_user_idx)
#     # Removed broadcast() from item_lookup, it's too big!
#     .join(item_lookup, "itemIndex") 
#     # Kept broadcast() here assuming your movie/game title database is small
#     .join(broadcast(enriched_content_df), "item_id", how="left") 
#     .select("item_id", "title", "original_language", "Genres")
# )
# history.show(10, truncate=False)

# # 4. Show Clean Recs (Filtering FIRST, then joining with Broadcast)
# print("\n[ New Suggestions Only ]")
# reco_10 = (clean_recommendations
#     .filter(col("userIndex") == random_user_idx)
#     .orderBy(col("score").desc())
#     .limit(10) 
#     # Removed broadcast() here too
#     .join(item_lookup, "itemIndex")
#     .join(broadcast(enriched_content_df), "item_id", how="left")
#     .select("item_id", "title", "original_language", "Genres")
# )
# reco_10.show(10, truncate=False)

In [16]:
# Pick a random user index from the CLEANED recommendations

random_user_idx = clean_recommendations.select("userIndex").sample(False, 0.1).limit(1).first()[0]

print(f"--- Recommendations for User Index: {random_user_idx} ---")


history = als_data.filter(col("userIndex") == random_user_idx).join(item_lookup, "itemIndex")

reco_10 = (clean_recommendations
    .filter(col("userIndex") == random_user_idx)
    .join(item_lookup, "itemIndex")
    .orderBy(col("score").desc())
    .limit(10))

# Show History
print("\n[ History ]")
history = history.join(enriched_content_df,"item_id", how="left").select("item_id", "title", "original_language", "Genres")
history.show(10, truncate=False)

# Show Clean Recs
print("\n[ New Suggestions Only ]")
reco_10 = reco_10.join(enriched_content_df,"item_id", how = "left").select("item_id", "title", "original_language", "Genres")
reco_10.show(10, truncate=False)

--- Recommendations for User Index: 191 ---

[ History ]


26/03/18 07:31:22 WARN DAGScheduler: Broadcasting large task binary with size 1770.1 KiB
26/03/18 07:31:23 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:34:06 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 28 for reason Executor for container container_1764236692086_5267_01_000045 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:34:06 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 21 for reason Executor for container container_1764236692086_5267_01_000038 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:36:36 WARN DAGScheduler: Broadcasting large task binary with size 174.3 MiB
26/03/18 07:37:21 WA

+--------------------------+--------+-----------------+-------+
|item_id                   |title   |original_language|Genres |
+--------------------------+--------+-----------------+-------+
|CHAUPAL_MOVIE_en_movie_894|Ikk Kudi|pa               |[]     |
|CHAUPAL_MOVIE_en_movie_878|NULL    |NULL             |NULL   |
|CHAUPAL_MOVIE_en_movie_874|Maa Jaye|pa               |[Drama]|
+--------------------------+--------+-----------------+-------+


[ New Suggestions Only ]


26/03/18 07:38:08 WARN DAGScheduler: Broadcasting large task binary with size 1770.1 KiB
26/03/18 07:38:09 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:43:17 WARN DAGScheduler: Broadcasting large task binary with size 174.3 MiB
26/03/18 07:43:24 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:43:31 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:43:31 WARN SparkConf: The configuration key 'spark.yarn.executor.failu

+---------------------------+--------------------------+-----------------+------------------------+
|item_id                    |title                     |original_language|Genres                  |
+---------------------------+--------------------------+-----------------+------------------------+
|CHAUPAL_MOVIE_en_movie_634 |Chhalla Mud Ke Nahi Aaya  |pa               |[]                      |
|CHAUPAL_MOVIE_en_movie_853 |Nikka Zaildar 4           |pa               |[Comedy, Drama, Romance]|
|CHAUPAL_MOVIE_en_movie_859 |NULL                      |NULL             |NULL                    |
|CHAUPAL_MOVIE_en_movie_860 |Madhaniyan                |pa               |[Romance, Drama]        |
|CHAUPAL_MOVIE_en_movie_864 |Ardaas Sarbat De Bhalle Di|pa               |[Drama, Family]         |
|ZEEFIVE_MOVIE_0-0-1z5873990|Godday Godday Chaa 2      |pa               |[Comedy, Drama]         |
|CHAUPAL_MOVIE_en_movie_856 |Furlow                    |pa               |[Comedy, Drama]         |


In [17]:
item_factors = model.itemFactors
item_features_final = (
    model.itemFactors
    .join(item_lookup, col("id") == col("itemIndex"))
    .select(
        "item_id",      # Your actual content ID/Name
        "itemIndex",    # The integer used by ALS
        "features"      # The latent vector [rank_1, rank_2, ...]
    )
)


### Show Recommendations by ID

In [18]:
# --- CONFIGURATION ---
TARGET_ITEM_ID = "SHEMAROOME_MOVIE_5b9fb803c1df41453400012b"  # Replace with the actual ID you want to look up
# ---------------------

# 1. Get the vector for your target item
target_row = item_features_final.filter(col("item_id") == TARGET_ITEM_ID).select("features").first()
enriched_content_df.filter(col("item_id") == TARGET_ITEM_ID).select("title", "ID", "item_id", "original_language", "Genres").show(truncate=False)
if not target_row:
    print(f"Item {TARGET_ITEM_ID} not found in the dataset.")
else:
    target_vector = target_row[0]
    num_features = len(target_vector)

    # 2. Calculate Similarity (Dot Product)
    # This creates a column where we multiply target[i] * features[i] for every dimension
    similar_items = (
        item_features_final
        .withColumn("similarity_score", 
            sum(col("features")[i] * target_vector[i] for i in range(num_features))
        )
        # 3. Filter out the item itself so you don't recommend the same thing
        .filter(col("item_id") != TARGET_ITEM_ID)
        .select("item_id", "similarity_score")
        .orderBy(col("similarity_score").desc())
        .limit(10)
        .join(enriched_content_df, "item_id", how = "left")
    )

    print(f"✨ Items most similar to: {TARGET_ITEM_ID}")
    similar_items.select("title", "ID", "item_id", "original_language", "Genres", "similarity_score").show(truncate=False)
# item_features_final.show(10, truncate=False)

26/03/18 07:50:03 WARN DAGScheduler: Broadcasting large task binary with size 174.3 MiB
26/03/18 07:50:07 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:50:43 WARN DAGScheduler: Broadcasting large task binary with size 174.0 MiB
26/03/18 07:51:05 WARN DAGScheduler: Broadcasting large task binary with size 175.9 MiB
26/03/18 07:51:15 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:51:23 WARN DAGScheduler: Broadcasting large task binary with size 176.0 MiB


+------------------+---------+-----------------------------------------+-----------------+-------------------------------------+
|title             |ID       |item_id                                  |original_language|Genres                               |
+------------------+---------+-----------------------------------------+-----------------+-------------------------------------+
|Amar Akbar Anthony|nfm_38022|SHEMAROOME_MOVIE_5b9fb803c1df41453400012b|hi               |[Action, Comedy, Drama, Music, Crime]|
+------------------+---------+-----------------------------------------+-----------------+-------------------------------------+

✨ Items most similar to: SHEMAROOME_MOVIE_5b9fb803c1df41453400012b


26/03/18 07:51:29 WARN DAGScheduler: Broadcasting large task binary with size 1771.4 KiB
26/03/18 07:51:30 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:52:33 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 40 for reason Executor for container container_1764236692086_5267_01_000067 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:52:33 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 43 for reason Executor for container container_1764236692086_5267_01_000070 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:52:33 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 45 f

+-------------------------------+-----------+--------------------------------------------------------------+-----------------+--------------------------------+--------------------+
|title                          |ID         |item_id                                                       |original_language|Genres                          |similarity_score    |
+-------------------------------+-----------+--------------------------------------------------------------+-----------------+--------------------------------+--------------------+
|3 Idiots                       |nfm_20453  |ULTRA_MOVIE_610902b73b971457336d8c6e                          |hi               |[Drama, Comedy]                 |0.044806411909826406|
|Pushpa - The Rise              |nfm_690957 |MINITV_MOVIE_amzn1.dv.gti.ac3855ce-6c21-4b6e-a8d8-dd0479dafda1|te               |[Action, Drama, Thriller]       |0.041291405365481544|
|Drishyam 2                     |nfm_1029827|MINITV_MOVIE_amzn1.dv.gti.2a67b1bd-15a9-414c-97e9-

## LightFM

In [19]:
import pandas as pd
import numpy as np
import ast
from lightfm.data import Dataset

# =====================================================================
# 1. FETCH DATA FROM SPARK TO PANDAS
# =====================================================================
print("Fetching interactions from Spark to Pandas...")
# WARNING: Ensure als_data is filtered/sampled down from the 100GB size!
interactions_pdf = als_data.select("userIndex", "itemIndex", "playtime_logged").toPandas()
print(f" -> Interactions loaded: {len(interactions_pdf)} rows")

print("Fetching items metadata from Spark to Pandas...")
items_pdf = (
    enriched_content_df.join(item_lookup, "item_id")
    .select("itemIndex", "item_id", "title", "original_language", "Genres")
    .dropDuplicates(["itemIndex"])
    .toPandas()
)
print(f" -> Items loaded: {len(items_pdf)} rows")

# =====================================================================
# 2. CLEAN DATA AND CAST TYPES
# =====================================================================
print("Cleaning missing values and casting integer types...")
items_pdf['Genres'] = items_pdf['Genres'].fillna('Unknown').astype(str)
items_pdf['original_language'] = items_pdf['original_language'].fillna('Unknown').astype(str)

interactions_pdf['userIndex'] = interactions_pdf['userIndex'].astype(int)
interactions_pdf['itemIndex'] = interactions_pdf['itemIndex'].astype(int)
items_pdf['itemIndex'] = items_pdf['itemIndex'].astype(int)

# =====================================================================
# 3. FILTER INTERACTIONS TO MATCH VALID ITEMS
# =====================================================================
print("Filtering orphan interactions...")
valid_item_indices = items_pdf['itemIndex'].unique()
interactions_pdf = interactions_pdf[interactions_pdf['itemIndex'].isin(valid_item_indices)]

valid_user_indices = interactions_pdf['userIndex'].unique()
print(f" -> Filtered to {len(interactions_pdf)} valid interactions.")

# =====================================================================
# 4. THE UNIVERSAL FEATURE FLATTENER
# =====================================================================
def extract_flat_features(lang, genre_data):
    """Safely extracts and flattens genres regardless of string/list/nested format."""
    features = [str(lang)]
    
    if isinstance(genre_data, str):
        if genre_data.startswith('['):
            try:
                genre_data = ast.literal_eval(genre_data)
            except:
                genre_data = genre_data.replace('[', '').replace(']', '').replace("'", "").replace('"', "")
                genre_data = [g.strip() for g in genre_data.split(',')]
        else:
            genre_data = [g.strip() for g in genre_data.split(',')]
            
    if isinstance(genre_data, list):
        for item in genre_data:
            if isinstance(item, list):
                features.extend([str(i).strip() for i in item])
            else:
                features.append(str(item).strip())
                
    return features

# Apply this logic ONCE to create a perfect list of features for every item
print("Processing item features...")
items_pdf['clean_features'] = items_pdf.apply(
    lambda row: extract_flat_features(row['original_language'], row['Genres']), axis=1
)
print(f" -> Sample clean features: {items_pdf['clean_features'].iloc[0]}")

# =====================================================================
# 5. EXTRACT UNIQUE FEATURES FOR LIGHTFM DATASET
# =====================================================================
print("Extracting global unique features...")
all_item_features = set()
for feature_list in items_pdf['clean_features']:
    all_item_features.update(feature_list)
    
all_item_features = list(all_item_features)
print(f" -> Total unique item features found: {len(all_item_features)}")

# =====================================================================
# 6. INITIALIZE AND FIT LIGHTFM DATASET
# =====================================================================
print("Fitting LightFM Dataset mapping...")
dataset = Dataset()
dataset.fit(
    users=valid_user_indices,
    items=valid_item_indices,
    item_features=all_item_features
)
num_users, num_items = dataset.interactions_shape()
print(f" -> Dataset successfully mapped: {num_users} users and {num_items} items.")

# =====================================================================
# 7. BUILD LIGHTFM MATRICES
# =====================================================================
print("Building Interactions matrix...")
(interactions, weights) = dataset.build_interactions(
    zip(
        interactions_pdf['userIndex'], 
        interactions_pdf['itemIndex'], 
        interactions_pdf['playtime_logged']
    )
)

print("Building Item Features matrix...")
item_features = dataset.build_item_features(
    (item_idx, features) 
    for item_idx, features in zip(
        items_pdf['itemIndex'], 
        items_pdf['clean_features'] # Passing our perfectly cleaned lists directly
    )
)

print("\nMatrices built successfully! Ready to train the model.")

Fetching interactions from Spark to Pandas...
 -> Interactions loaded: 56281838 rows
Fetching items metadata from Spark to Pandas...


26/03/18 08:00:38 WARN DAGScheduler: Broadcasting large task binary with size 1770.0 KiB
26/03/18 08:00:39 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 08:05:50 WARN DAGScheduler: Broadcasting large task binary with size 174.3 MiB
26/03/18 08:06:29 WARN DAGScheduler: Broadcasting large task binary with size 174.0 MiB
26/03/18 08:06:49 WARN DAGScheduler: Broadcasting large task binary with size 175.9 MiB
26/03/18 08:06:58 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 08:07:04 WARN DAGScheduler: Broadcasting large task binary with size 175.9 MiB
26/03/18 08:07:23 WARN DAGScheduler: Broadcasting large task bina

 -> Items loaded: 15372 rows
Cleaning missing values and casting integer types...
Filtering orphan interactions...
 -> Filtered to 44745072 valid interactions.
Processing item features...
 -> Sample clean features: ['te', 'Action']
Extracting global unique features...
 -> Total unique item features found: 88
Fitting LightFM Dataset mapping...


26/03/18 08:08:39 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 56 for reason Executor for container container_1764236692086_5267_01_000097 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.


 -> Dataset successfully mapped: 5218529 users and 15372 items.
Building Interactions matrix...


26/03/18 08:08:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 59 for reason Executor for container container_1764236692086_5267_01_000101 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:08:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 57 for reason Executor for container container_1764236692086_5267_01_000099 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:08:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 54 for reason Executor for container container_1764236692086_5267_01_000095 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:08:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 53 for reason Executor for container container_1764236692086_5

Building Item Features matrix...

Matrices built successfully! Ready to train the model.


In [20]:
from lightfm import LightFM

# Initialize the model (WARP is best for implicit ranking)
lfm_model = LightFM(
    loss='warp',
    no_components=20,
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6
)

print("Training LightFM model...")
lfm_model.fit(
    interactions,
    item_features=item_features,
    sample_weight=weights,
    epochs=10,
    num_threads=4 
)
print("Training complete!")

Training LightFM model...
Training complete!


In [21]:
# Get LightFM's internal mappings
user_id_map, user_feature_map, item_id_map, item_feature_map = dataset.mapping()

# Create a reverse dictionary to map LightFM's internal item index back to your Spark itemIndex
inverse_item_id_map = {v: k for k, v in item_id_map.items()}

def recommend_for_user(target_user_index, model, interactions, item_features, num_recs=10):
    # Convert Spark userIndex to LightFM internal user index
    internal_user_id = user_id_map.get(target_user_index)
    if internal_user_id is None:
        return "User not found."
    
    # Get internal indices of items the user has already watched
    known_positives = interactions.tocsr()[internal_user_id].indices
    
    # Predict scores for all items
    n_items = len(item_id_map)
    scores = model.predict(
        internal_user_id, 
        np.arange(n_items), 
        item_features=item_features
    )
    
    # Rank the items
    top_items_indices = np.argsort(-scores)
    
    # Filter out known positives
    top_items_indices = [idx for idx in top_items_indices if idx not in known_positives]
    
    # Get top N Spark itemIndexes
    top_spark_item_indexes = [inverse_item_id_map[idx] for idx in top_items_indices[:num_recs]]
    
    # Return the readable dataframe
    return items_pdf[items_pdf['itemIndex'].isin(top_spark_item_indexes)][['item_id', 'title', 'original_language', 'Genres']]


In [22]:
# print(f"--- LightFM Recommendations for User Index: {random_user_idx} ---")
# print(recommend_for_user(random_user_idx, lfm_model, interactions, item_features))

## ALS vs LighFM comparisons

### Random User

In [23]:
# Pick a random user index from the CLEANED recommendations

random_user_idx = (
    clean_recommendations
    .select("userIndex")
    .distinct() # Optional: Ensures you don't pick a user with heavy weighting more often
    .orderBy(F.rand()) 
    .limit(1)
    .first()[0]
)

print(f"--- Recommendations for User Index: {random_user_idx} ---")


history = als_data.filter(col("userIndex") == random_user_idx).join(item_lookup, "itemIndex")

reco_10 = (clean_recommendations
    .filter(col("userIndex") == random_user_idx)
    .join(item_lookup, "itemIndex")
    .orderBy(col("score").desc())
    .limit(10))

# Show History
print("\n[ History ]")
history = history.join(enriched_content_df,"item_id", how="left").select("item_id", "title", "original_language", "Genres")
history.show(10, truncate=False)

# Show Clean Recs
print("\n[ ALS Suggestions ]")
reco_10 = reco_10.join(enriched_content_df,"item_id", how = "left").select("item_id", "title", "original_language", "Genres")
reco_10.show(10, truncate=False)

print(f"--- LightFM Recommendations for User Index: {random_user_idx} ---")
# Get the Pandas DataFrame from your function
lightfm_pandas_df = recommend_for_user(random_user_idx, lfm_model, interactions, item_features)

# Convert it to Spark and print using the exact same formatting!
lightfm_spark_df = spark.createDataFrame(lightfm_pandas_df)
lightfm_spark_df.select("item_id", "title", "original_language", "Genres").show(10, truncate=False)

26/03/18 08:22:45 WARN TaskSetManager: Lost task 48.0 in stage 915.0 (TID 29506) (dprc-wynk-prd-data-ds-machine-learning-common-sw-c59r.c.prj-wynk-prd-ds-svc-01.internal executor 61): java.lang.StackOverflowError
	at java.base/java.io.ObjectInputStream$BlockDataInputStream.readByte(ObjectInputStream.java:3376)
	at java.base/java.io.ObjectInputStream.readHandle(ObjectInputStream.java:1798)
	at java.base/java.io.ObjectInputStream.readClassDesc(ObjectInputStream.java:1862)
	at java.base/java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2201)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1687)
	at java.base/java.io.ObjectInputStream.defaultReadFields(ObjectInputStream.java:2496)
	at java.base/java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2390)
	at java.base/java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2228)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1687)
	at java.base/ja

--- Recommendations for User Index: 4740519 ---

[ History ]


26/03/18 08:24:20 WARN DAGScheduler: Broadcasting large task binary with size 1770.1 KiB
26/03/18 08:24:22 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 08:25:24 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 63 for reason Executor for container container_1764236692086_5267_01_000114 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:29:35 WARN DAGScheduler: Broadcasting large task binary with size 174.3 MiB
26/03/18 08:30:31 WARN DAGScheduler: Broadcasting large task binary with size 174.0 MiB
26/03/18 08:30:54 WARN DAGScheduler: Broadcasting large task binary with size 175.9 MiB
26/03/18 08:30:56 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 31 for reason Exec

+---------------------------+-------------------+-----------------+----------------+
|item_id                    |title              |original_language|Genres          |
+---------------------------+-------------------+-----------------+----------------+
|ZEEFIVE_MOVIE_0-0-1z5485690|Once Upon Two Times|hi               |[Drama, Romance]|
|ZEEFIVE_MOVIE_0-0-1z5646239|NULL               |NULL             |NULL            |
|ZEEFIVE_MOVIE_0-0-1z5485648|Safed              |hi               |[Drama]         |
+---------------------------+-------------------+-----------------+----------------+


[ ALS Suggestions ]


26/03/18 08:31:22 WARN DAGScheduler: Broadcasting large task binary with size 1770.1 KiB
26/03/18 08:31:25 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 08:33:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 83 for reason Executor for container container_1764236692086_5267_01_000140 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:33:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 78 for reason Executor for container container_1764236692086_5267_01_000131 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:33:42 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 68 f

+----------------------------+------------------------+-----------------+------------------------------------+
|item_id                     |title                   |original_language|Genres                              |
+----------------------------+------------------------+-----------------+------------------------------------+
|ZEEFIVE_MOVIE_0-0-1z5831123 |NULL                    |NULL             |NULL                                |
|ZEEFIVE_MOVIE_0-0-1z5857652 |Saali Mohabbat          |hi               |[Drama]                             |
|ZEEFIVE_TVSHOW_0-6-3284     |Sunflower               |hi               |[Comedy, Crime]                     |
|ZEEFIVE_TVSHOW_0-6-4z5860309|NULL                    |NULL             |NULL                                |
|ZEEFIVE_MOVIE_0-0-1z5149852 |Forensic                |hi               |[Crime, Thriller]                   |
|ZEEFIVE_MOVIE_0-0-1z5334341 |Mrs. Undercover         |hi               |[Action, Comedy, Thriller]          |
|

+-----------------------------+-------------------------------+-----------------+---------------------------------------------+
|item_id                      |title                          |original_language|Genres                                       |
+-----------------------------+-------------------------------+-----------------+---------------------------------------------+
|SONYLIV_VOD_TVSHOW_1700000084|Taarak Mehta Ka Ooltah Chashmah|hi               |['Comedy', 'Drama', 'Family', 'Kids', 'Soap']|
|ZEEFIVE_TVSHOW_0-6-4z5612471 |Vasudha                        |hi               |['Drama']                                    |
|ZEEFIVE_TVSHOW_0-6-4z5793364 |Ganga Mai Ki Betiyan           |hi               |['Drama', 'Family']                          |
|HUNGAMA_TVSHOW_124548472     |Prayagraj Ki Love Story        |hi               |['Drama']                                    |
|SONYLIV_VOD_TVSHOW_1700000725|Maharani                       |hi               |['Drama']              

### Particular user

In [24]:
# Pick a random user index from the CLEANED recommendations

# 1. Define the item_lookup correctly (Overwriting the old Window version)


# random_user_idx = (
#     clean_recommendations
#     .select("userIndex")
#     .distinct() # Optional: Ensures you don't pick a user with heavy weighting more often
#     .orderBy(F.rand()) 
#     .limit(1)
#     .first()[0]
# )

from pyspark.sql import functions as F

# 1. Create a distinct mapping from your indexed data
user_mapping = indexed_data.select("userId", "userIndex").distinct()

# 2. Filter for your specific ID
target_id = "ib-xPv06AjuYOE4BI0"
result = user_mapping.filter(F.col("userId") == target_id).collect()

if result:
    print(f"UserIndex: {int(result[0]['userIndex'])}")
    target_user_idx = int(result[0]['userIndex'])
    # target_user_id = "ib-xPv06AjuYOE4BI0"
    print(f"--- Recommendations for User Index: {target_user_idx} ---")


    history = als_data.filter(col("userIndex") == target_user_idx).join(item_lookup, "itemIndex")

    reco_10 = (clean_recommendations
        .filter(col("userIndex") == target_user_idx)
        .join(item_lookup, "itemIndex")
        .orderBy(col("score").desc())
        .limit(10))

    # Show History
    print("\n[ History ]")
    history = history.join(enriched_content_df,"item_id", how="left").select("item_id", "title", "original_language", "Genres")
    history.show(10, truncate=False)

    # Show Clean Recs
    print("\n[ ALS Suggestions ]")
    reco_10 = reco_10.join(enriched_content_df,"item_id", how = "left").select("item_id", "title", "original_language", "Genres")
    reco_10.show(10, truncate=False)

    print(f"--- LightFM Recommendations for User Index: {target_user_idx} ---")
    # Get the Pandas DataFrame from your function
    lightfm_pandas_df = recommend_for_user(target_user_idx, lfm_model, interactions, item_features)

    # Convert it to Spark and print using the exact same formatting!
    lightfm_spark_df = spark.createDataFrame(lightfm_pandas_df)
    lightfm_spark_df.select("item_id", "title", "original_language", "Genres").show(10, truncate=False)
else:
    print("User ID not found.", result)



26/03/18 08:39:53 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 107 for reason Executor for container container_1764236692086_5267_01_000169 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:39:53 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 106 for reason Executor for container container_1764236692086_5267_01_000168 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:39:53 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 134 for reason Executor for container container_1764236692086_5267_01_000199 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:39:53 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 94 for reason Executor for container container_176423669208

UserIndex: 3845409
--- Recommendations for User Index: 3845409 ---

[ History ]


26/03/18 08:46:17 WARN DAGScheduler: Broadcasting large task binary with size 1770.1 KiB
26/03/18 08:46:20 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 08:47:50 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 156 for reason Executor for container container_1764236692086_5267_01_000230 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:47:50 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 144 for reason Executor for container container_1764236692086_5267_01_000218 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:47:50 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 14

+-----------------------------------+---------------+-----------------+-----------------------------+
|item_id                            |title          |original_language|Genres                       |
+-----------------------------------+---------------+-----------------+-----------------------------+
|LIONSGATEPLAY_MOVIE_12STRONG0Y2018M|12 Strong      |en               |[War, Drama, Action, History]|
|SONYLIV_VOD_MOVIE_1000131111       |Black Hawk Down|en               |[Action, War, History]       |
+-----------------------------------+---------------+-----------------+-----------------------------+


[ ALS Suggestions ]


26/03/18 08:52:30 WARN DAGScheduler: Broadcasting large task binary with size 1770.1 KiB
26/03/18 08:52:31 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 08:54:55 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 179 for reason Executor for container container_1764236692086_5267_01_000259 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:54:55 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 184 for reason Executor for container container_1764236692086_5267_01_000267 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 08:54:55 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 18

+-----------------------------------------------------+---------------------------------+-----------------+------------------------------------+
|item_id                                              |title                            |original_language|Genres                              |
+-----------------------------------------------------+---------------------------------+-----------------+------------------------------------+
|LIONSGATEPLAY_MOVIE_SNITCHY2013M                     |Snitch                           |en               |[Thriller, Drama, Action]           |
|LIONSGATEPLAY_MOVIE_JOHNWICKY2014M                   |John Wick                        |en               |[Action, Thriller]                  |
|LIONSGATEPLAY_MOVIE_AFTERBURNY2025M                  |Afterburn                        |en               |[Science Fiction, Action, Comedy]   |
|LIONSGATEPLAY_MOVIE_REDSONJAY2025M                   |Red Sonja                        |en               |[Adventure, Action, Fan

In [ ]:
watch_history_df.filter(F.col("userId") == "ib-xPv06AjuYOE4BI0").show(20, truncate=False)

+------------------+-----------------------------------+-------------------+----------+
|userId            |item_id                            |total_play_time_sec|day       |
+------------------+-----------------------------------+-------------------+----------+
|ib-xPv06AjuYOE4BI0|LIONSGATEPLAY_MOVIE_12STRONG0Y2018M|2.0                |2026-03-12|
|ib-xPv06AjuYOE4BI0|LIONSGATEPLAY_MOVIE_12STRONG0Y2018M|1321.0             |2026-01-05|
|ib-xPv06AjuYOE4BI0|SONYLIV_VOD_MOVIE_1000131111       |1.0                |2025-12-26|
+------------------+-----------------------------------+-------------------+----------+



26/03/18 09:46:11 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 11 for reason Executor for container container_1764236692086_5267_01_000028 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 09:51:11 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 19 for reason Executor for container container_1764236692086_5267_01_000036 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 09:56:14 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 27 for reason Executor for container container_1764236692086_5267_01_000044 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 09:56:14 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 34 for reason Executor for container container_1764236692086_5

In [ ]:
# Pick a random user index from the CLEANED recommendations

# 1. Define the item_lookup correctly (Overwriting the old Window version)


# random_user_idx = (
#     clean_recommendations
#     .select("userIndex")
#     .distinct() # Optional: Ensures you don't pick a user with heavy weighting more often
#     .orderBy(F.rand()) 
#     .limit(1)
#     .first()[0]
# )

from pyspark.sql import functions as F

# 1. Create a distinct mapping from your indexed data
user_mapping = indexed_data.select("userId", "userIndex").distinct()

# 2. Filter for your specific ID
target_id = "YNkVt86cnYgIYKBOM0"
result = user_mapping.filter(F.col("userId") == target_id).collect()

if result:
    print(f"UserIndex: {int(result[0]['userIndex'])}")
    target_user_idx = int(result[0]['userIndex'])
    # target_user_id = "ib-xPv06AjuYOE4BI0"
    print(f"--- Recommendations for User Index: {target_user_idx} ---")


    history = als_data.filter(col("userIndex") == target_user_idx).join(item_lookup, "itemIndex")

    reco_10 = (clean_recommendations
        .filter(col("userIndex") == target_user_idx)
        .join(item_lookup, "itemIndex")
        .orderBy(col("score").desc())
        .limit(10))

    # Show History
    print("\n[ History ]")
    history = history.join(enriched_content_df,"item_id", how="left").select("item_id", "title", "original_language", "Genres")
    history.show(10, truncate=False)

    # Show Clean Recs
    print("\n[ ALS Suggestions ]")
    reco_10 = reco_10.join(enriched_content_df,"item_id", how = "left").select("item_id", "title", "original_language", "Genres")
    reco_10.show(10, truncate=False)

    print(f"--- LightFM Recommendations for User Index: {target_user_idx} ---")
    # Get the Pandas DataFrame from your function
    lightfm_pandas_df = recommend_for_user(target_user_idx, lfm_model, interactions, item_features)

    # Convert it to Spark and print using the exact same formatting!
    lightfm_spark_df = spark.createDataFrame(lightfm_pandas_df)
    lightfm_spark_df.select("item_id", "title", "original_language", "Genres").show(10, truncate=False)
else:
    print("User ID not found.", result)



26/03/18 10:38:43 WARN DAGScheduler: Broadcasting large task binary with size 174.3 MiB
26/03/18 10:39:25 WARN DAGScheduler: Broadcasting large task binary with size 174.0 MiB
26/03/18 10:40:01 WARN DAGScheduler: Broadcasting large task binary with size 200.9 MiB


UserIndex: 2920218
--- Recommendations for User Index: 2920218 ---

[ History ]


26/03/18 10:40:18 WARN DAGScheduler: Broadcasting large task binary with size 1770.1 KiB
26/03/18 10:40:19 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 10:41:22 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 199 for reason Executor for container container_1764236692086_5267_01_000288 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 10:41:22 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 193 for reason Executor for container container_1764236692086_5267_01_000281 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 10:41:22 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 19

+------------------------------------+------------------------------+-----------------+-----------------------------------------------+
|item_id                             |title                         |original_language|Genres                                         |
+------------------------------------+------------------------------+-----------------+-----------------------------------------------+
|ZEEFIVE_TVSHOW_0-6-4z5887209        |NULL                          |NULL             |NULL                                           |
|LIONSGATEPLAY_MOVIE_MENINBLACKY1997M|Men in Black                  |en               |[Action, Adventure, Comedy, Science Fiction]   |
|ZEEFIVE_MOVIE_0-0-1z5878368         |Ek Deewane Ki Deewaniyat      |hi               |[Romance, Drama, Thriller]                     |
|ZEEFIVE_MOVIE_0-0-1z5874437         |Boonie Bears: The Wild Life   |zh               |[Animation, Comedy, Science Fiction, Adventure]|
|SONYLIV_VOD_MOVIE_1000007756        |Men in Bla

26/03/18 10:46:50 WARN DAGScheduler: Broadcasting large task binary with size 1770.1 KiB
26/03/18 10:46:51 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 10:49:15 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 210 for reason Executor for container container_1764236692086_5267_01_000305 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 10:49:15 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 218 for reason Executor for container container_1764236692086_5267_01_000315 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 10:49:15 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 21

+---------------------------------------------------------------+-------------------------------+-----------------+------------------------------------+
|item_id                                                        |title                          |original_language|Genres                              |
+---------------------------------------------------------------+-------------------------------+-----------------+------------------------------------+
|SONYLIV_VOD_TVSHOW_1700001014                                  |Indian Idol                    |hi               |[Reality]                           |
|ZEEFIVE_MOVIE_0-0-1z5895209                                    |Mastiii 4                      |hi               |[Comedy]                            |
|SONYLIV_VOD_TVSHOW_1790006756                                  |Wheel Of Fortune               |hi               |[Drama, Reality, Comedy, Family]    |
|SONYLIV_VOD_TVSHOW_1700000084                                  |Taarak Mehta Ka O

+--------------------------------------------------------------+------------+-----------------+------------------------------------------------------+
|item_id                                                       |title       |original_language|Genres                                                |
+--------------------------------------------------------------+------------+-----------------+------------------------------------------------------+
|LIONSGATEPLAY_MOVIE_GODZILLAY1998M                            |गॉडज़िला    |en               |['Science Fiction', 'Action', 'Thriller']             |
|LIONSGATEPLAY_MOVIE_SNITCHY2013M                              |Snitch      |en               |['Thriller', 'Drama', 'Action']                       |
|LIONSGATEPLAY_MOVIE_SPIDERMAN2Y2004M                          |Spider-Man 2|en               |['Action', 'Adventure', 'Science Fiction']            |
|LIONSGATEPLAY_MOVIE_SPIDERMAN3Y2007M                          |Spider-Man 3|en               

26/03/18 10:55:45 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 241 for reason Executor for container container_1764236692086_5267_01_000340 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 10:55:45 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 246 for reason Executor for container container_1764236692086_5267_01_000345 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 10:55:45 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 247 for reason Executor for container container_1764236692086_5267_01_000346 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 10:55:45 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 236 for reason Executor for container container_17642366920